In [40]:
BUDGET_COMMUNES = "models/bronze/budget_per_compte_communes.sql"

Ouvrir les fichiers


In [ ]:
with open(BUDGET_COMMUNES, "r") as file:
    sql_query = file.read()


Importer duckdb


In [42]:
import duckdb

con = duckdb.connect("dev.duckdb")
result = con.sql(sql_query)


Afficher les données


In [43]:
result.limit(10).df()

,annee,siret,code_departement,code_insee,code_region,type_compte,solde_debiteur,solde_crediteur,solde,code_geo
0,2010,21010001200017,01,1,82,depenses,329891.87,5999.38,323892.49,011
1,2010,21010010300014,01,10,82,depenses,1160202.00,590.23,1159611.77,0110
2,2010,21010100200017,01,100,82,depenses,101188.40,0.00,101188.40,01100
3,2010,21010101000010,01,101,82,depenses,138281.77,0.00,138281.77,01101
4,2010,21010102800012,01,102,82,depenses,344885.95,1636.00,343249.95,01102
5,2010,21010103600015,01,103,82,depenses,714615.11,14354.48,700260.63,01103
6,2010,21010104400019,01,104,82,depenses,375347.08,5108.66,370238.42,01104
7,2010,21010105100014,01,105,82,depenses,682435.30,15753.15,666682.15,01105
8,2010,21010106900016,01,106,82,depenses,129276.31,0.00,129276.31,01106
9,2010,21010107700019,01,107,82,depenses,109671.87,18.00,109653.87,01107


Aggréger les données


In [48]:
df = con.sql("""
WITH CTE_1 AS (
    SELECT
        EXER AS annee,
        IDENT AS siret,
        NDEPT AS code_departement,
        INSEE AS code_insee,
        CREGI AS code_region,
        type_compte AS type_compte,
        SD AS solde_debiteur,
        SC AS solde_crediteur,
        (SD - SC) AS solde,
        (CAST(NDEPT AS TEXT) || CAST(INSEE AS TEXT)) AS code_geo
    FROM 'pipeline_inputs/budget_per_compte_communes.csv'
    WHERE type_compte = 'primes d assurances'
    -- WHERE type_compte NOT IN ('depenses','produits','dettes financieres')
)

SELECT
    annee,
    SUM(solde) AS total
FROM CTE_1
GROUP BY annee
ORDER BY annee
""").df()

import pandas as pd

pd.options.display.float_format = "{:,.0f}".format

df


,annee,total
0,2010,"482,359,234"
1,2011,"490,821,760"
2,2012,"508,132,287"
3,2013,"529,739,592"
4,2014,"549,842,911"
5,2015,"556,256,072"
6,2016,"566,093,149"
7,2017,"555,261,091"
8,2018,"536,345,215"
9,2019,"458,434,412"
